<a href="https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. Two paper findings + my methodology questions

## **Finding 1: The Anatomy of Growing Content**

**Source:** *The State of AI-Driven SEO*, March 2026, page 6.

The paper reports that growing content averaged approximately 3,180 words and 184 days in age, while declining content averaged approximately 2,311 words and 230 days in age. The paper describes this as an observational comparison and notes that growing pages tend to be longer, younger, and slightly better positioned.

**My methodology question:** Were the growing and declining groups compared using consistent reporting windows and comparable inclusion criteria? Could differences in topic, search demand, or existing visibility also help explain the observed differences in word count and age?

**Why I would ask this:** The comparison identifies useful associations, but it does not establish that increasing word count or refreshing a page will cause traffic to grow. Understanding other differences between the groups would help readers interpret the finding appropriately.

## **Finding 2: The Freshness Multiplier**

**Source:** *The State of AI-Driven SEO*, March 2026, page 9.

The paper reports a growth-to-decline ratio of approximately 7.88:1 for the 31–90-day freshness window. It also explicitly cautions that the 361+ day bucket has a ratio of 283:1 because it contains 283 growing pages and only one declining page.

**My methodology question:** What are the sample sizes and uncertainty for each freshness bucket, and how sensitive are the reported ratios to small numbers of declining pages? For the comparison between recently refreshed and older content, were similar pages that were not refreshed examined over a comparable period?

**Why I would ask this:** The paper already acknowledges that the 361+ day ratio is unstable. Checking group sizes and comparison design helps distinguish a useful observed pattern from a result that may be sensitive to the selected sample. A comparison of refreshed and unrefreshed pages would also help assess alternative explanations without assuming that refreshes caused the observed difference.

##*Reflection**

These questions are intended to understand the evidence and its limits, not to dismiss the research. The paper provides useful portfolio-level observations, while causal claims about the effects of content refresh would require stronger evidence. I will apply the same standard to my own model by checking its label construction, validation design, feature timing, and errors.

## 2. My model under an honest split (before/after)

In Week 5, I evaluated a rule-based baseline and Logistic Regression using a client-grouped split. The training set contained 24 clients and the test set contained 8 different clients, with no client overlap.

The grouped test results were:

- Week-4 baseline: Precision@20 = 0.60; Precision@50 = 0.56; average precision = 0.520.
- Logistic Regression: Precision@20 = 0.30; Precision@50 = 0.38; average precision = 0.514.
- Test positive-label rate: 0.517.

For this audit, I will compare a random row split with a client-grouped split using the same features, baseline rules, model, and ranking metrics. A random split allows pages from the same client to appear in training and testing, while a client-grouped split evaluates generalization to unseen clients.

The comparison measures sensitivity to the validation design. It does not establish that either split predicts the success of a future content refresh.

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# Load the dataset if it is not already in memory
if "df" not in globals():
    df = pd.read_csv("/content/content_refresh_anonymized.csv")

data = df.copy()
data["is_down"] = (data["trend_direction"] == "down").astype(int)

# Week-4 baseline rules: median thresholds
impression_median = data["impressions_90d"].median()
ctr_median = data["ctr"].median()

data["baseline_score"] = (
    40 * (data["days_since_last_update"] >= 180).astype(int)
    + 40 * (data["impressions_90d"] >= impression_median).astype(int)
    + 20 * (data["ctr"] < ctr_median).astype(int)
)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

feature_cols = numeric_features + categorical_features

# Same 25% test size and fixed seed for both validation designs
all_indices = np.arange(len(data))

random_train_idx, random_test_idx = train_test_split(
    all_indices,
    test_size=0.25,
    random_state=42,
    stratify=data["is_down"],
)

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

group_train_idx, group_test_idx = next(
    group_splitter.split(data, groups=data["client_id"])
)

def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    order = np.argsort(np.asarray(scores))[::-1][:k]
    return np.asarray(y_true)[order].mean()

def evaluate_split(split_name, train_idx, test_idx):
    train = data.iloc[train_idx].copy()
    test = data.iloc[test_idx].copy()

    X_train = train[feature_cols]
    y_train = train["is_down"]
    X_test = test[feature_cols]
    y_test = test["is_down"]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ])

    model.fit(X_train, y_train)
    model_scores = model.predict_proba(X_test)[:, 1]
    baseline_scores = test["baseline_score"].to_numpy()

    return {
        "split": split_name,
        "train_rows": len(train),
        "test_rows": len(test),
        "train_clients": train["client_id"].nunique(),
        "test_clients": test["client_id"].nunique(),
        "client_overlap": len(
            set(train["client_id"]) & set(test["client_id"])
        ),
        "test_positive_rate": y_test.mean(),
        "baseline_P@20": precision_at_k(y_test, baseline_scores, 20),
        "model_P@20": precision_at_k(y_test, model_scores, 20),
        "baseline_P@50": precision_at_k(y_test, baseline_scores, 50),
        "model_P@50": precision_at_k(y_test, model_scores, 50),
        "baseline_AP": average_precision_score(y_test, baseline_scores),
        "model_AP": average_precision_score(y_test, model_scores),
    }

comparison = pd.DataFrame([
    evaluate_split("Random row split", random_train_idx, random_test_idx),
    evaluate_split("Client-grouped split", group_train_idx, group_test_idx),
])

display(comparison.round(3))

,split,train_rows,test_rows,train_clients,test_clients,client_overlap,test_positive_rate,baseline_P@20,model_P@20,baseline_P@50,model_P@50,baseline_AP,model_AP
0,Random row split,22500,7500,32,31,31,0.542,0.75,0.8,0.72,0.74,0.583,0.651
1,Client-grouped split,22885,7115,24,8,0,0.517,0.60,0.3,0.56,0.38,0.520,0.514


###**Before/after validation comparison**

Under the random row split, Logistic Regression achieved Precision@20 of 0.80, Precision@50 of 0.74, and average precision of 0.651. The rule-based baseline achieved 0.75, 0.72, and 0.583, respectively.

Under the client-grouped split, Logistic Regression achieved Precision@20 of 0.30, Precision@50 of 0.38, and average precision of 0.514. The baseline achieved 0.60, 0.56, and 0.520.

The random split included 31 clients in both training and test data. The grouped split had no client overlap, with 24 training clients and 8 held-out test clients. The model's ranking metrics were substantially lower under the grouped split, and it no longer outperformed the baseline.

This gap suggests that the random-split evaluation may overstate performance when generalizing to unseen clients. It does not prove that client memorization is the sole cause. The grouped test set contains only eight clients, and its positive-label rate (0.517) differs from the random test set (0.542). These results are directional evidence about validation sensitivity, not evidence that content refresh causes traffic growth.

## 3. Leakage audit

I reviewed the model features for direct use of the target, target-derived fields, identifiers, existing baseline scores, and potentially overlapping performance windows.

The model excludes `trend_direction`, `trend_pct`, the last-30-day metrics, the previous-30-day metrics, `content_id`, and `client_id` from its direct feature inputs. The Week-4 baseline score is used for comparison only, not as a Logistic Regression feature.

However, excluding these columns does not establish that all leakage has been eliminated. The exact definition and observation window of `trend_direction` need to be verified. Aggregate performance metrics may overlap with the period used to assign the trend label. I will document this uncertainty rather than claim the model is fully leakage-free.

In [6]:
# Audit the exact features used in Week 5

target_related = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

identifiers_and_scores = [
    "content_id",
    "client_id",
    "baseline_score",
]

print("Target-related columns included as model features:")
print([c for c in feature_cols if c in target_related])

print("\nIdentifiers or baseline scores included as model features:")
print([c for c in feature_cols if c in identifiers_and_scores])

print("\nFeatures used by Logistic Regression:")
print(feature_cols)

print("\nTarget construction:")
print("is_down = (trend_direction == 'down')")

print("\nTrend label distribution:")
display(
    df["trend_direction"]
      .value_counts(dropna=False)
      .rename_axis("trend_direction")
      .to_frame("rows")
)

print("\nPerformance metric window checks:")
window_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

display(df[window_cols].describe().T)

Target-related columns included as model features:
[]

Identifiers or baseline scores included as model features:
[]

Features used by Logistic Regression:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier']

Target construction:
is_down = (trend_direction == 'down')

Trend label distribution:


,rows
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



Performance metric window checks:


,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.0,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.0,7.00,4178.0
sessions_90d,30000.0,37.066633,107.069131,1.0,2.0,7.0,27.00,4345.0
impressions_last_30d,30000.0,1429.058733,5643.852081,0.0,10.0,139.0,768.00,238796.0
clicks_last_30d,30000.0,4.933867,23.929393,0.0,0.0,0.0,2.00,1176.0
sessions_last_30d,30000.0,14.114267,39.036371,0.0,0.0,3.0,12.00,1081.0
impressions_prev_30d,30000.0,1783.078500,6150.429511,0.0,19.0,210.0,1143.00,218786.0
clicks_prev_30d,30000.0,5.435100,28.358673,0.0,0.0,0.0,2.00,1627.0
sessions_prev_30d,30000.0,10.283000,42.578003,0.0,1.0,2.0,7.00,4247.0


### Leakage audit conclusion

The feature-list audit found no direct inclusion of the target columns (`trend_direction` and `trend_pct`), client/content identifiers, or the Week-4 baseline score in the Logistic Regression inputs.

However, I cannot yet certify the model as leakage-free. The dataset contains 90-day aggregate performance metrics, while the target is an observed trend category. If the trend label is calculated from recent-versus-previous-period performance, the 90-day aggregates may overlap with the information used to construct the label.

Before making a stronger claim, I would verify the label-generation logic and the feature observation dates. A future-outcome model would require all features to be available before the prediction date and its target window to occur strictly afterwards. The current analysis is a classification of an observed label, not a validated forecast of future refresh outcomes.

## 4. Claim rewrite

### **Original claim that goes beyond the evidence**

“Logistic Regression predicts which pages need refreshing and can help improve their traffic.”

###**Revised claim**

“On this anonymized dataset, Logistic Regression classified the observed `down` trend label. Its performance varied by validation design. Under a random row split, average precision was 0.651; under a client-grouped split, it was 0.514. On the grouped test set, the Week-4 rule-based baseline slightly exceeded the model in average precision (0.520 versus 0.514) and performed better at Precision@20 and Precision@50.

These results suggest that the random-split evaluation may overstate performance for unseen clients. The model has not been shown to predict the causal effect of a content refresh or to guarantee traffic improvements. Because the exact trend-label construction and temporal overlap have not been verified, the findings should be treated as directional decision-support evidence.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.